# Regression Testing for a Conversational Agent

Every prompt change is a potential regression. Classic unit tests don't help here: the
output is non-deterministic natural language, so `assert output == expected` is useless.
The answer is a **test suite of simulated conversations** that is scored automatically —
so a prompt edit turns the suite red the same way a broken function turns a unit test red.

**System under test:** a phone-support routing agent for a fictional property manager
("Stadtquartier Wohnservice"). A tenant calls, the agent asks a few questions and routes
the call to the correct department.

**Three LLM roles — keep them apart, this is the core idea:**

| Role | Defined in | Job |
| --- | --- | --- |
| **System under test** | §2 `route_conversation` | the agent we want to protect from regressions |
| **Test agent** | §3 `test_agent_message` | plays the tenant, driven by a persona — replaces a human tester |
| **Judge** | §5 `evaluate_result` | LLM-as-Judge, scores the agent's tone 1–5 against a rubric |

**The loop, per scenario:**

```
scenario ──▶ run_test_conversation ──▶ evaluate_result ──▶ pass / fail
 (persona)     (test agent  ⇄  agent)     (3 assertions)
```

**A scenario passes only if all three assertions hold:**

1. `routing_correct` — the call landed in the expected department (a *functional* assertion)
2. `within_budget` — the agent needed no more than `max_turns` tenant turns (an *efficiency* assertion)
3. `friendliness >= min_friendliness` — judged tone clears the bar (a *qualitative* assertion)

**How to use this notebook:** run §1–§6 top to bottom for a green baseline, then run §7 to
deliberately damage the routing prompt, re-run §6 and watch the suite catch it, then §8 to repair it.

**Requires** `OPENAI_API_KEY` in a `.env` file next to this notebook. Every run costs real
API calls — roughly 3 model calls per conversation turn plus one judge call per scenario.

##  1. Setup and Test Scenarios

Client setup plus the two pieces of static data the whole suite is built on:

* **`DEPARTMENTS`** — the routing taxonomy. This dict is the single source of truth: it is
  injected into the routing prompt *and* used to render the department name back to the
  tenant. Adding a department here is enough to teach the agent about it.
* **`TEST_SCENARIOS`** — the test cases. Each one is a plain dict; think of it as one
  parametrised test:

| Key | Meaning |
| --- | --- |
| `id` | stable test id for the report (`TC-001`, …) |
| `description` | what this case is probing |
| `persona` | system prompt for the **test agent** — who the tenant is and how they behave |
| `expected_department` | the assertion. Two special values: `"auth-failed"` = we expect the call to be stopped at authentication; `None` = any department is acceptable |
| `max_turns` | turn budget — an agent that needs 9 questions is broken even if it routes correctly |
| `min_friendliness` | lower bound on the judge's 1–5 score |

**Coverage is deliberate, not random** — the six cases probe different failure modes:
happy path (TC-001), calm/direct (TC-002), hostile tone (TC-003), security /
authentication (TC-004), under-specified input where guessing is the failure (TC-005),
and a semantic trap where the obvious keyword points at the wrong department (TC-006).

In [1]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()   # reads OPENAI_API_KEY from the .env file next to this notebook

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
model = "gpt-5-mini"   # same model for agent, test agent and judge — cheap and fast enough for a suite

# Routing taxonomy: key = machine-readable department id the model must return,
# value = human-readable description. Both halves get injected into the routing prompt,
# so the description is what actually teaches the model the boundaries between departments.
DEPARTMENTS = {
    "rental-contracts":     "Rental Contracts — lease questions, renewals, amendments",
    "terminations-moveout": "Terminations & Move-out — cancellations, move-out dates, deposit returns",
    "tenant-complaints":    "Tenant Complaints — noise, neighbour disputes, general grievances",
    "energy-heating":       "Energy & Heating — heating failures, hot water, utility billing",
    "repairs-maintenance":  "Repairs & Maintenance — broken fixtures, structural damage, general repairs",
}

# The test suite. One dict = one test case. Fields: see the markdown cell above.
TEST_SCENARIOS = [
    {
        # Happy path: unambiguous issue, cooperative tenant. If this fails, everything is broken.
        "id": "TC-001",
        "description": "Clear heating issue, cooperative tenant",
        "persona": (
            "You are Lisa Müller, a tenant at Berliner Str. 12, Berlin. "
            "Your heating stopped working 2 days ago. You are polite but worried because it's cold. "
            "Give your name and address when asked. Keep answers short and factual."
        ),
        "expected_department": "energy-heating",
        "max_turns": 5,
        "min_friendliness": 3,
    },
    {
        # Second happy path in a different department — guards against the agent collapsing
        # onto one favourite category.
        "id": "TC-002",
        "description": "Lease termination, calm and direct",
        "persona": (
            "You are Thomas Bauer, a tenant at Hauptstr. 7, Hamburg. "
            "You want to end your lease. You are calm and businesslike. "
            "Give your name and address when asked."
        ),
        "expected_department": "terminations-moveout",
        "max_turns": 5,
        "min_friendliness": 2,   # businesslike caller — we accept a flatter, professional tone
    },
    {
        # Tone test: the tenant is frustrated. The agent must stay warm (friendliness >= 3)
        # instead of mirroring the irritation — a classic regression after prompt edits.
        "id": "TC-003",
        "description": "Noise complaint, frustrated tone",
        "persona": (
            "You are Emre Yilmaz, a tenant at Gartenweg 3, Munich. "
            "Your upstairs neighbour makes noise every night and you haven't slept in a week. "
            "You are frustrated and slightly impatient. Give your details when asked."
        ),
        "expected_department": "tenant-complaints",
        "max_turns": 6,   # a venting caller needs one extra turn
        "min_friendliness": 3,
    },
    {
        # Security test: an unverifiable address must END the call, not get routed anyway.
        # Note the persona explicitly refuses to correct itself — otherwise a helpful test
        # agent would "fix" the address and the test would silently stop testing anything.
        "id": "TC-004",
        "description": "Auth failure — wrong address",
        "persona": (
            "You are Sandra Koch. You want to report a broken window. "
            "When asked for your address, say 'Musterstr. 99, Berlin' — this is not a real tenant address. "
            "Do not correct it even if the agent says it can't be verified."
        ),
        "expected_department": "auth-failed",   # special value: we expect the call to end here
        "max_turns": 3,
        "min_friendliness": 3,   # being rejected is not an excuse for a cold tone
    },
    {
        # Under-specification: "I have a problem with my apartment" fits four departments.
        # The failure mode is *guessing*; the persona withholds detail until asked directly.
        "id": "TC-005",
        "description": "Ambiguous issue — agent must ask, not guess",
        "persona": (
            "You are Klaus Werner, a tenant at Friedrichstr. 45, Berlin. "
            "Your opening message is just: 'I have a problem with my apartment.' "
            "Only give more details if the agent asks a direct question. Be cooperative."
        ),
        "expected_department": None,            # any correct department is acceptable — we test that the agent asks first
        "max_turns": 6,
        "min_friendliness": 3,
    },
    {
        # Semantic trap: the word "cancel" screams terminations-moveout, but the tenant is
        # cancelling an ENERGY contract. Keyword matching fails here; real comprehension passes.
        "id": "TC-006",
        "description": "Tricky phrasing — sounds like terminations but is energy",
        "persona": (
            "You are Yuki Tanaka, a tenant at Rosenweg 8, Frankfurt. "
            "You want to cancel your energy supply contract, not your apartment lease. "
            "Say: 'I want to cancel my energy contract.' Give your details when asked."
        ),
        "expected_department": "energy-heating",
        "max_turns": 5,
        "min_friendliness": 2,
    },
]

## 2. The Routing Agent (Standalone Version)

**This is the system under test.** Everything else in the notebook exists to exercise these
functions. "Standalone" = pulled out of the production stack so it can be called directly
by the harness, without a phone line or a web server in the way.

Two design points worth pausing on:

* **Structured output instead of free text.** `client.beta.chat.completions.parse(...)` with
  a Pydantic `response_format` forces the model to return a validated `RoutingDecision`
  object. The test can then assert on `.department` — a field, not a sentence. Without
  this you'd be regex-ing prose, and *that* parsing would be what breaks.
* **`confidence` is part of the contract.** The model reports how sure it is, and §4 only
  accepts `"medium"`/`"high"` as a final answer. That is what lets the agent keep asking
  questions instead of guessing (the behaviour TC-005 tests).

`ROUTING_SYSTEM_PROMPT` is the fragile artefact this whole suite protects — §7 breaks
exactly this string.

⚠️ `check_auth()` is defined here for completeness but is **not** wired into the runner —
§4 uses a crude string heuristic instead, deliberately, so the auth path stays deterministic
and free. See the note in §4.

In [2]:
from pydantic import BaseModel

# --- Output schemas -------------------------------------------------------
# Pydantic models double as (a) the JSON schema the model is forced to fill in and
# (b) the typed object the tests assert against. Field names/types ARE the contract.

class AuthResult(BaseModel):
    verified: bool
    customer_name: str
    reason: str

class RoutingDecision(BaseModel):
    department: str        # must be one of the DEPARTMENTS keys
    routing_reason: str    # why — makes a wrong decision debuggable instead of mysterious
    issue_summary: str     # short recap handed over to the receiving department
    confidence: str        # "low" / "medium" / "high" — gates the hand-off in §4

# --- The prompt under test ------------------------------------------------
# Built from DEPARTMENTS so the taxonomy is never duplicated. The `k: v` join is the
# important part: keys alone tell the model *what to output*, the descriptions tell it
# *when to choose what*. §7 removes the descriptions to show how much work they do.
ROUTING_SYSTEM_PROMPT = (
    "You are a routing agent at Stadtquartier Wohnservice, a residential property management company.\n"
    "Based on the conversation transcript, decide which department should handle this tenant's issue.\n\n"
    "Available departments:\n"
    + "\n".join(f"- {k}: {v}" for k, v in DEPARTMENTS.items())
    + "\n\nChoose exactly one department key from the list above. "
    "In routing_reason, explain specifically why this department is the right fit."
)

def route_conversation(transcript: list[dict]) -> RoutingDecision | None:
    """Takes a list of {"role": "tenant"/"agent", "text": "..."} dicts, returns routing decision."""
    # Flatten the dialogue into one labelled block. The router is stateless: it sees the
    # whole conversation as a single user message, not as a chat history.
    transcript_text = "\n".join(f"{t['role'].upper()}: {t['text']}" for t in transcript)
    try:
        # .parse() = structured outputs. The response is schema-validated by the SDK,
        # so .parsed is either a real RoutingDecision or an exception — never half-valid prose.
        response = client.beta.chat.completions.parse(
            model=model,
            messages=[
                {"role": "system", "content": ROUTING_SYSTEM_PROMPT},   # ← read at CALL time,
                                                                       #   so §7/§8 can swap it live
                {"role": "user", "content": f"Conversation:\n{transcript_text}"}
            ],
            response_format=RoutingDecision
        )
        return response.choices[0].message.parsed
    except Exception:
        # Swallow API/validation errors and return None. The runner treats None as
        # "no routing yet" and simply asks another question — a failed call must not
        # crash the suite, it should show up as a FAIL row in the report.
        return None

def check_auth(name: str, address: str) -> AuthResult:
    # NOTE: currently unused — the runner in §4 uses a string heuristic instead.
    # Kept as the realistic version: an LLM standing in for a tenant-database lookup.
    response = client.beta.chat.completions.parse(
        model=model,
        messages=[
            {"role": "system", "content": (
                "You simulate a tenant database check. "
                "If the address has a recognisable street name and number, mark verified. "
                "Clearly fictional or incomplete addresses (like 'Musterstr. 99') are not verified."
            )},
            {"role": "user", "content": f"Name: {name}\nAddress: {address}"}
        ],
        response_format=AuthResult
    )
    return response.choices[0].message.parsed

## 3. Test Agent (the Simulated Tenant)

The test agent replaces the human who would otherwise have to call the hotline six times per
prompt change. It is a *second* LLM whose system prompt is the scenario's `persona`, so it
stays in character across turns and produces realistic, varied phrasing — including the
messy, indirect wording real callers use.

**The one subtlety in this cell is the role flip.** Our transcript stores roles from the
*agent's* point of view (`"agent"` / `"tenant"`). The test agent lives on the other side of
the table, so when we replay history into it, the mapping inverts:

| transcript role | role sent to the test agent |
| --- | --- |
| `"tenant"` (= the test agent's own past lines) | `assistant` |
| `"agent"` (= what the support agent said) | `user` |

Get this wrong and the persona starts answering its own questions — a bug that produces
plausible-looking transcripts and silently invalidates every result.

**Trade-off to be aware of:** a simulated tenant is not a real one. It is more cooperative
and more grammatical than reality, so a green suite means "no regression", not "works with
real users".

In [3]:
def test_agent_message(persona: str, conversation_history: list[dict]) -> str:
    """Given a persona and the conversation so far, generate the next customer message."""
    # The persona becomes the system prompt -> the model *is* this tenant for the whole call.
    messages = [
        {"role": "system", "content": (
            f"{persona}\n\n"
            "You are participating in a customer support call. "
            "Respond naturally as this person would. "
            "Keep each response short — 1–3 sentences maximum. "   # keeps turns realistic and cheap
            "Do not break character."                              # stops it from answering as an AI assistant
        )}
    ]
    # Add conversation history: agent messages become 'user' from the test agent's perspective
    # (and the tenant's own past lines become 'assistant'). Mirror image of the transcript roles.
    for turn in conversation_history:
        role = "assistant" if turn["role"] == "tenant" else "user"
        messages.append({"role": role, "content": turn["text"]})

    # Explicit nudge so the model produces the NEXT line instead of commenting on the transcript.
    messages.append({"role": "user", "content": "What do you say next?"})

    response = client.chat.completions.create(model=model, messages=messages)
    return response.choices[0].message.content.strip()

## 4. Conversation Runner

Drives one scenario end to end and returns the raw material for scoring. No assertions here —
this cell only *produces* a conversation; §5 judges it. Keeping those two apart means you can
re-score an old transcript with a new rubric without paying for the conversation again.

**Turn loop, in order:**

1. The agent opens with a fixed greeting (constant, so it never costs a call and never varies).
2. The **test agent** replies as the tenant.
3. **Auth gate** — if the tenant's text looks unverifiable, the call is terminated. This is
   the exit TC-004 expects.
4. **Routing attempt** — only once the conversation has ≥ 5 entries (greeting + ~2 exchanges).
   Accepted only at `confidence` medium/high; a low-confidence answer is discarded and the
   loop continues. That gate is the mechanism behind TC-005: *ask, don't guess*.
5. **Otherwise** a receptionist prompt asks one short clarifying question, and we loop.

**Two shortcuts you should read as teaching scaffolding, not production code:**

* The auth check is a substring test (`"Musterstr"` / `"99"`), not `check_auth()`. It is
  deterministic and free, which keeps TC-004 stable — but `"99"` matches *any* occurrence,
  including a legitimate house number 99 or the year 1999. In a real system this is the
  step-auth span from session 2.
* Routing and clarifying live in an `if/else`, so from turn 3 onward the agent *only* tries
  to route — it stops asking questions. The loop is capped at `max_turns + 2` so a
  never-terminating conversation still ends (and then trips the turn-budget assertion).

In [ ]:
# Fixed opener: deterministic, costs nothing, and gives every transcript the same starting point.
AGENT_GREETING = (
    "Thank you for calling Stadtquartier Wohnservice. "
    "My name is Chappy, how can I help you today?"
)

def run_test_conversation(scenario: dict) -> dict:
    """Runs one full test conversation. Returns transcript + routing result."""
    conversation = []        # list of {"role": "agent"|"tenant", "text": str}
    routing_result = None    # stays None if the agent never reaches a confident decision
    auth_failed = False      # True once the call is terminated at the auth gate

    # Agent opens
    conversation.append({"role": "agent", "text": AGENT_GREETING})

    # Hard cap on the loop: max_turns + 2 gives the agent a little slack, while still
    # guaranteeing termination. Going over max_turns is not an error here — it is recorded
    # and fails `within_budget` in the evaluator.
    for turn_num in range(scenario["max_turns"] + 2):  # small buffer
        # Test agent responds as customer
        customer_msg = test_agent_message(scenario["persona"], conversation)
        conversation.append({"role": "tenant", "text": customer_msg})

        # Simple auth extraction (look for name + address pattern)
        # In a real system this would be the step-auth span from session 2.
        # Deliberately crude: a deterministic substring test keeps TC-004 stable and free.
        # Caveat — "99" matches any occurrence (house number 99, "1999", …), so this would
        # produce false rejections in production. Swap in check_auth() for the realistic version.
        tenant_messages = " ".join(t["text"] for t in conversation if t["role"] == "tenant")
        if "Musterstr" in tenant_messages or "99" in tenant_messages:
            auth_failed = True
            conversation.append({
                "role": "agent",
                "text": "I'm sorry, I wasn't able to verify your details with that address. Please contact us directly."
            })
            break   # security stop: no routing after a failed verification — this is what TC-004 asserts

        # After enough context (3+ turns), attempt routing.
        # len(conversation) >= 5 == greeting + 2 full exchanges, i.e. don't route on the
        # opening line alone.
        if len(conversation) >= 5:
            routing_result = route_conversation(conversation)
            # Confidence gate: a "low" answer is thrown away and we loop for more context.
            # This is the guard against confident nonsense on vague input (TC-005).
            if routing_result and routing_result.confidence in ("medium", "high"):
                agent_reply = (
                    f"Thank you. I'm routing you to our {DEPARTMENTS.get(routing_result.department, routing_result.department)} team. "
                    f"{routing_result.routing_reason}"
                )
                conversation.append({"role": "agent", "text": agent_reply})
                break   # successful hand-off ends the call
        else:
            # Early turns: a cheap receptionist prompt that asks ONE short question
            # (name / address / what's wrong) to build up the context the router needs.
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": "You are a friendly receptionist. Ask for the tenant's name, address, or clarify their issue. Keep it to one short question."},
                    {"role": "user", "content": "\n".join(f"{t['role']}: {t['text']}" for t in conversation)}
                ]
            )
            agent_reply = response.choices[0].message.content.strip()
            conversation.append({"role": "agent", "text": agent_reply})

    # Raw result — no pass/fail here. Scoring happens in the evaluator, so the same
    # transcript can be re-judged later with a different rubric.
    return {
        "scenario_id": scenario["id"],
        "transcript": conversation,
        "routing": routing_result,
        "auth_failed": auth_failed,
    }

## 5. The Evaluator — Three Assertions

Turns one transcript into one pass/fail row. Three checks, and **all three must pass**:

| # | Check | How | Kind |
| --- | --- | --- | --- |
| 1 | `routing_correct` | compare `routing.department` with `expected_department` | deterministic |
| 2 | `within_budget` | count tenant turns ≤ `max_turns` | deterministic |
| 3 | `friendliness` | LLM-as-Judge against `FRIENDLINESS_RUBRIC`, ≥ `min_friendliness` | model-scored |

**Note the split:** two cheap deterministic assertions, one model-based. Use the judge only
for what genuinely cannot be checked with code (tone, helpfulness, faithfulness). Judging
things you could `==` is slow, costly and adds noise.

**Three branches for the expected department** — this is where the special values from §1 pay off:

* `"auth-failed"` → correct means the call was *stopped*, `auth_failed is True`.
* `None` (TC-005) → any routing counts; the real assertion for that scenario is the turn budget,
  because guessing early is exactly what we're testing against.
* anything else → exact string match on the department key.

**Making the judge work:** the rubric anchors the scale with concrete descriptions at 1/3/5
(rather than "rate 1–5", which drifts), and `response_format={"type": "json_object"}` forces
parseable JSON. The judge also returns a `reason` — used here only for eyeballing, but it is
what makes a surprising score debuggable.

In [5]:
# Anchored rubric: describing what 1, 3 and 5 mean keeps scores comparable across runs.
# A bare "rate friendliness 1-5" drifts between calls and makes the threshold meaningless.
FRIENDLINESS_RUBRIC = """
Score the AGENT's tone on a scale of 1 to 5:
  1 = cold or dismissive
  3 = professional but flat
  5 = warm and empathetic
Return JSON: {"score": <1-5>, "reason": "<one sentence>"}
"""

def evaluate_result(scenario: dict, result: dict) -> dict:
    transcript = result["transcript"]
    routing = result["routing"]
    auth_failed = result["auth_failed"]

    # --- Assertion 1: routing (deterministic) -----------------------------
    if scenario["expected_department"] == "auth-failed":
        # TC-004: success = the call was stopped at auth, NOT routed somewhere.
        routing_correct = auth_failed
    elif scenario["expected_department"] is None:
        routing_correct = True  # TC-005: any routing is fine, we check separately
                                # (the real assertion there is the turn budget: ask, don't guess)
    else:
        # Exact match on the department key. `routing is None` (API error, or the agent
        # never got confident) counts as incorrect.
        routing_correct = (routing is not None and routing.department == scenario["expected_department"])

    # --- Assertion 2: efficiency (deterministic) --------------------------
    # Count only tenant turns: how many times did we have to bother the caller?
    turn_count = len([t for t in transcript if t["role"] == "tenant"])
    within_budget = turn_count <= scenario["max_turns"]

    # --- Assertion 3: tone (LLM-as-Judge) ---------------------------------
    # The judge sees the whole transcript, both sides — tone is only readable in context.
    convo_text = "\n".join(f"{t['role'].upper()}: {t['text']}" for t in transcript)
    judge_response = client.chat.completions.create(
        model="gpt-5-mini",   # hardcoded on purpose: pinning the judge means scores stay
                              # comparable even if you swap the agent's `model` above
        response_format={"type": "json_object"},   # guarantees json.loads() below won't choke
        messages=[
            {"role": "system", "content": FRIENDLINESS_RUBRIC},
            {"role": "user", "content": convo_text}
        ]
    )
    friendliness_result = json.loads(judge_response.choices[0].message.content)
    friendliness_score = friendliness_result["score"]
    # friendliness_result["reason"] is available here too — useful when a score looks wrong.

    # AND over all three: one red assertion fails the scenario.
    passed = routing_correct and within_budget and friendliness_score >= scenario["min_friendliness"]

    # One flat dict per scenario = one row in the report. Keep both the verdict and the
    # evidence (actual department, turn count, the model's reason) so a FAIL is diagnosable.
    return {
        "id": scenario["id"],
        "description": scenario["description"],
        "expected": scenario["expected_department"] or "any",
        "actual": routing.department if routing else ("auth-failed" if auth_failed else "no-routing"),
        "routing_correct": routing_correct,
        "turns": turn_count,
        "within_budget": within_budget,
        "friendliness": friendliness_score,
        "routing_reason": routing.routing_reason if routing else "—",
        "passed": passed,
    }

## 6. Test Runner and Report

`run_suite()` is the CI job: for every scenario, run the conversation, evaluate it, print a
live PASS/FAIL line. `print_report()` renders the table plus the four numbers you actually
track over time:

* **passed / total** — the headline; anything below 100 % is a regression to explain
* **routing accuracy** — the functional metric, isolated from tone and turn count
* **avg turns** — efficiency; creeping upward means the agent is getting chattier
* **avg friendliness** — tone; the number that silently degrades when you optimise a prompt for accuracy

Record these four after every prompt change — that comparison *is* the regression test.
Because the models are non-deterministic, expect ±1 scenario of jitter between runs: a
single flip is noise, a metric moving in one direction across runs is a real regression.

**Run this cell now for your baseline, then again after §7 and after §8.** The
`if __name__ == "__main__":` guard is always true in a notebook, so the suite executes on run;
it exists so this file also works as a plain script. Full suite ≈ 6 conversations × several
turns — the slowest and most expensive cell in the notebook.

In [9]:
def run_suite(scenarios: list) -> list:
    """Run every scenario end to end, print live progress, collect the result rows."""
    results = []
    for scenario in scenarios:
        print(f"Running {scenario['id']}: {scenario['description']}...")
        conversation_result = run_test_conversation(scenario)   # §4: produce the transcript
        eval_result = evaluate_result(scenario, conversation_result)   # §5: score it
        results.append(eval_result)
        # Live feedback per scenario — a long suite shouldn't be a silent black box.
        status = "✅ PASS" if eval_result["passed"] else "❌ FAIL"
        print(f"  {status} — routing: {eval_result['actual']} | turns: {eval_result['turns']} | friendliness: {eval_result['friendliness']}/5")
    return results

def print_report(results: list):
    """Fixed-width table + the four metrics to track across prompt versions."""
    print("\n" + "=" * 75)
    # Column widths are padded to match the rows below — keeps the table aligned in a terminal.
    print(f"{'ID':<8} {'Expected':<22} {'Actual':<22} {'Turns':>5} {'Friend':>7} {'Result':>6}")
    print("-" * 75)
    for r in results:
        status = "PASS" if r["passed"] else "FAIL"
        # Expected vs. actual side by side: a wrong department is readable at a glance.
        print(
            f"{r['id']:<8} {r['expected']:<22} {r['actual']:<22} "
            f"{r['turns']:>5} {r['friendliness']:>7} {status:>6}"
        )

    # --- Aggregate metrics: write these down before and after every prompt change ---
    passed = sum(1 for r in results if r["passed"])
    total = len(results)
    routing_acc = sum(1 for r in results if r["routing_correct"]) / total   # functional quality alone
    avg_turns = sum(r["turns"] for r in results) / total                    # efficiency trend
    avg_friendliness = sum(r["friendliness"] for r in results) / total      # tone trend

    print("=" * 75)
    print(f"SUMMARY: {passed}/{total} passed | routing accuracy: {routing_acc:.0%} | avg turns: {avg_turns:.1f} | avg friendliness: {avg_friendliness:.1f}/5")

# True inside a notebook, so running this cell runs the suite; the guard keeps the file
# usable as a plain `python regression_testing.py` script as well.
if __name__ == "__main__":
    results = run_suite(TEST_SCENARIOS)
    print_report(results)

Running TC-001: Clear heating issue, cooperative tenant...
  ❌ FAIL — routing: energy-heating | turns: 7 | friendliness: 4/5
Running TC-002: Lease termination, calm and direct...
  ❌ FAIL — routing: terminations-moveout | turns: 7 | friendliness: 4/5
Running TC-003: Noise complaint, frustrated tone...
  ❌ FAIL — routing: tenant-complaints | turns: 8 | friendliness: 4/5
Running TC-004: Auth failure — wrong address...
  ✅ PASS — routing: auth-failed | turns: 1 | friendliness: 3/5
Running TC-005: Ambiguous issue — agent must ask, not guess...
  ✅ PASS — routing: repairs-maintenance | turns: 5 | friendliness: 3/5
Running TC-006: Tricky phrasing — sounds like terminations but is energy...
  ❌ FAIL — routing: energy-heating | turns: 7 | friendliness: 4/5

ID       Expected               Actual                 Turns  Friend Result
---------------------------------------------------------------------------
TC-001   energy-heating         energy-heating             7       4   FAIL
TC-002   ter

##  7. Break Something — Watch the Suite Go Red

The demo. A suite that has never gone red proves nothing — you don't know whether it can
detect anything at all.

This cell rebinds `ROUTING_SYSTEM_PROMPT` to a damaged version. Two things are removed, and
the damage looks harmless — a shorter prompt, same departments:

1. **the department descriptions** (`- {k}` instead of `- {k}: {v}`) — the model now sees five
   bare ids and has to guess what `terminations-moveout` covers. Watch TC-006 ("cancel my
   **energy** contract") slide into terminations, and ambiguous cases scatter.
2. **the `routing_reason` instruction** — the field still exists in the schema, so it gets
   filled with something, but nothing steers it. Debuggability degrades quietly, without
   any error.

Rebinding works because `route_conversation()` reads the global at call time (§2) — no
function needs redefining.

**Now re-run §6 and compare all four metrics against your baseline.** Expect routing accuracy
to drop while friendliness stays flat: the agent is still perfectly polite while sending
people to the wrong department. That dissociation is the point — a tone metric alone would
have shown green.

In [7]:
# Break the routing prompt:
# Same departments, shorter prompt — looks like a harmless cleanup, is a regression.
ROUTING_SYSTEM_PROMPT = (
    "You are a routing agent. Route the tenant to a department. "   # generic role, no company context
    "Available departments:\n"
    + "\n".join(f"- {k}" for k in DEPARTMENTS.keys())   # keys only: the descriptions that
                                                        # defined the boundaries are gone
    # ponytail: removed department descriptions and routing_reason instruction intentionally
)
# route_conversation() reads this global on every call -> the next §6 run uses the broken prompt.

## 8. Fix It

Restore the full prompt (descriptions + `routing_reason` instruction) and re-run §6 — the
suite should return to your baseline. That round trip *green → red → green* is what
validates the harness itself: the suite reacts to a real change, and it reacts in the metric
that matters.

**The takeaway for your own agents:**

* Write the scenarios **before** you start tuning prompts — otherwise "better" is just a feeling.
* Cover failure modes on purpose (security, ambiguity, semantic traps, hostile tone), not just the happy path.
* Assert deterministically wherever you can; save the LLM judge for what code cannot check.
* Track several metrics — accuracy, efficiency, tone — because prompt changes trade them against each other.
* Re-run the suite on every prompt edit and keep the numbers. A prompt change without a suite run is an untested deploy.

In [8]:
# Restore the original prompt: department descriptions back in (k: v) and the explicit
# routing_reason instruction returned. Re-run §6 — the metrics should match the baseline.
ROUTING_SYSTEM_PROMPT = (
    "You are a routing agent at Stadtquartier Wohnservice, a residential property management company.\n"
    "Based on the conversation transcript, decide which department should handle this tenant's issue.\n\n"
    "Available departments:\n"
    + "\n".join(f"- {k}: {v}" for k, v in DEPARTMENTS.items())   # descriptions = the actual routing rules
    + "\n\nChoose exactly one department key. "
    "In routing_reason, explain specifically why this department is the right fit."
)